# Rag Assignment
Authors: Gianluca Amato and David Farrugia

--- short notebook description and overview of sections

## 1. Setup

### 1.1 Imports

This section imports all the required Python libraries used throughout the notebook and a fixed random seed is set for both Python’s `random` module and NumPy for reproducible results across multiple executions of the notebook.

In [1]:
import time
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

import requests
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt_tab')

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gianl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gianl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### 1.2 Path Setup and Input Validation

This section is responsible for setting up the directory structure required by the notebook and validating the presence of essential input data files.

Output and temporary dataset directories are created automatically if they do not already exist, ensuring that files can be written without manual setup at all stages of the notebook. The use of `Pathlib` allows for clean and platform independent path management.

The input dataset directory and the expected CSV file are then defined. A helper function is initialised to validate the input CSV path by checking that the file exists, is a valid file (not a directory), and has the correct `.csv` extension.

This validation step helps catch configuration or file errors early, preventing silent failures or misleading results later

In [4]:
# Create Dataset Directories if they don't exist
DATA_OUTPUT_DIR = Path("./Datasets/Outputs")
DATA_TEMP_DIR = Path("./Datasets/TempDatasets")

for directory in [DATA_OUTPUT_DIR, DATA_TEMP_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

# Paths
DATA_INPUT_DIR = Path("./Datasets/Inputs")
FSA_PATH = DATA_INPUT_DIR / "FSA_data.csv"

def validate_csv_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path.resolve()}")
    if not path.is_file():
        raise ValueError(f"{label} exists but is not a file: {path.resolve()}")
    if path.suffix.lower() != ".csv":
        raise ValueError(f"{label} is not a CSV file: {path.resolve()}")
    print(f"[OK] {label}: {path.resolve()}")
    
validate_csv_path(FSA_PATH, "FSA Data")

[OK] FSA Data: C:\Users\gianl\Documents\GitHub\ICS5111-MiningLargeScaleData_Project\Datasets\Inputs\FSA_data.csv


## 2. Data Processing

This section defines the core text preprocessing function used throughout the data processing stage of the pipeline.

An English stopword list is first initialised using the NLTK library. Stopwords are common function words (such as “the”, “and”, or “is”) that typically carry little discriminative value in term-based retrieval models and are therefore removed during sparse preprocessing.

A preprocessing function is then defined to normalise and filter text data. The function performs basic cleaning by converting text to lowercase, tokenising it into individual terms, removing stopwords, and discarding non-alphabetic tokens. This reduces noise while preserving content-bearing terms that are informative for sparse retrieval techniques such as TF–IDF or BM25.

An optional case is provided to retain four digit numeric tokens corresponding to years. This is particularly relevant in financial and economic datasets, where temporal information can play an important role in retrieval and interpretation.

The output of this function is a string of filtered tokens, which is later stored alongside the original raw text. This helps us maintaining both representations and allows the pipeline to apply different retrieval strategies using the appropriate format of data.


In [5]:
# Get English stop words set
stop_words = set(stopwords.words('english'))
    
# Function to preprocess text for sparse representation
def tokenize_and_filter(text, keep_years=False):
    # Guard against NaN / non-string values
    if not isinstance(text, str):
        return ""
    
    # Lowercase, tokenize, remove stopwords and non-alphabetic tokens
    tokens = word_tokenize(text.lower())
    
    filtered_tokens = []
    
    # Filter tokens
    for word in tokens:
        # keep alphabetic words
        if word.isalpha() and word not in stop_words:
            filtered_tokens.append(word)
            
            # optionally keep years (4-digit numbers)
        elif keep_years == True and word.isdigit() and len(word) == 4:
            filtered_tokens.append(word)
        
    # Join tokens back into a single string
    return " ".join(filtered_tokens)

#### 2.1 [Kaggle Financial Sentiment Analysis Dataset](https://www.kaggle.com/sbhatti/financial-sentiment-analysis)

This subsection processes the Kaggle Financial Sentiment Analysis dataset. The dataset is first loaded from disk and annotated with a source identifier to track its origin once multiple datasets are combined.

Each sentence is then preprocessed using the shared preprocessing function, producing a cleaned `processed_text` representation while preserving the original raw text. This dual representation allows different retrieval strategies to operate on the most appropriate form of the data.

A year extraction step is applied to the processed text by scanning for valid four-digit numeric tokens within a predefined range. When present, the extracted year is stored as structured metadata, enabling temporal filtering or analysis in later stages of the pipeline. This was mainly done since the 2nd dataset automatically returns a year column.

In [6]:
# Load dataset from disk
kaggle_dataset = pd.read_csv(FSA_PATH)

# Apply preprocessing to dataset for each sentence and reorder columns for better readability
kaggle_dataset["source"] = "kaggle"
kaggle_dataset["processed_text"] = kaggle_dataset["Sentence"].apply(tokenize_and_filter, keep_years=True)

# Create a year column based on processed text (if any year exists in the text)
def extract_year(text, min_year=2000, max_year=2025):
    tokens = text.split()
    
    for token in tokens:
        # Check if token is a 4-digit number
        if token.isdigit() and len(token) == 4:
            # Convert token to integer
            year = int(token)
            
            # Check if year is within valid range
            if min_year <= year <= max_year:
                return year
        
    return None

kaggle_dataset["year"] = kaggle_dataset["processed_text"].apply(extract_year)

kaggle_dataset = kaggle_dataset.rename(columns={"Sentence": "text", "Sentiment": "sentiment"})
kaggle_dataset = kaggle_dataset[["source", "text", "processed_text", "year", "sentiment"]]

print("Kaggle FSA Processed Data Sample:")
display(kaggle_dataset.head(10))

# Save processed dataset to disk
KAGGLE_OUTPUT_PATH = DATA_TEMP_DIR / "kaggle_fsa_dataset.csv"
kaggle_dataset.to_csv(KAGGLE_OUTPUT_PATH, index=False)

Kaggle FSA Processed Data Sample:


,source,text,processed_text,year,sentiment
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,NaN,positive
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,NaN,negative
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,2010.0,positive
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,NaN,neutral
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,NaN,neutral
5,kaggle,$SPY wouldn't be surprised to see a green close,spy would surprised see green close,NaN,positive
6,kaggle,Shell's $70 Billion BG Deal Meets Shareholder ...,shell billion bg deal meets shareholder skepti...,NaN,negative
7,kaggle,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,ssh communications security corp stock exchang...,2008.0,negative
8,kaggle,Kone 's net sales rose by some 14 % year-on-ye...,kone net sales rose first nine months 2008,2008.0,positive
9,kaggle,The Stockmann department store will have a tot...,stockmann department store total floor space s...,NaN,neutral


### 2.2 [World Bank Open Data](https://data.worldbank.org/)

This subsection retrieves structured economic indicators from the World Bank Open Data API. A base API endpoint is defined and used to fetch indicator values for selected countries over a specified time range.

A helper function is implemented to handle data retrieval. For each country, the function creates the appropriate API request, specifies the desired date range, and requests the data in JSON format. Error handling is applied to detect and halt execution in the case of invalid or failed requests.

Returned records are then parsed and transformed into a tabular format containing the country name, year, and indicator value. Requests are issued sequentially with a short delay between calls to avoid exceeding API rate limits. The output is a unified DataFrame containing all retrieved records.

>**[NOTE]:**
>**The API connection can occasionally timeout. If this happens please rerun the notebook again. Thank you** 

In [7]:
WB_API_URL = "https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"

# Function to fetch data from World Bank API
def fetch_indicator_data(indicator, countries, start_year, end_year, per_page=2000, rate_limit_delay=0.2):
    '''
    This fuction fetches data from World Bank API for given indicator and countries within specified year range.
    
    Parameters:
    - indicator (str): World Bank indicator code
    - contries (list): List of country codes
    - start_year (int): Start year for data
    - end_year (int): End year for data
    - per_page (int): Number of records per page
    - rate_limit_delay (float): Delay between requests to avoid rate limiting
    
    Returns:
    - DataFrame with columns ['country', 'year', 'value']
    '''
    all_records = []
    
    # Loop through countries one by one
    for country in countries:
        # Construct API URL
        url = WB_API_URL.format(
            country=country,
            indicator=indicator,
        )
        
        # Set query parameters
        params = {
            "date": f"{start_year}:{end_year}",
            "format": "json",
            "per_page": per_page,
        }
        
        # Make API request
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status() # Raise error for bad responses
        data = response.json()
    
        if len(data) < 2 or not data[1]:
            # No returned data for this country, skip
            time.sleep(rate_limit_delay)
            continue
        
        # Append records for this country
        for entry in data[1]:
            all_records.append({
                "country": entry["country"]["value"],
                "year": entry["date"],
                "value": entry["value"]
            })
    
        # Wait before next request so we don't spam the API
        time.sleep(rate_limit_delay)
    
    return pd.DataFrame(all_records)

This subsection retrieves economic indicators from the World Bank Open Data API for a predefined set of countries and years. A small collection of indicators relevant to financial and economic analysis is specified, along with the target countries and time range.

For each indicator, data is fetched for all countries using the previously defined API retrieval function. The results are annotated with a descriptive indicator name and combined into a single dataset covering multiple economic dimensions. The dataset is then labelled with a source identifier and its columns are reordered into a consistent structure.

In [8]:
# Congifure parameters for data fetching
COUNTRIES = {
    "MLT": "Malta",
    "ITA": "Italy",
    "DEU": "Germany",
    "FRA": "France",
    "ESP": "Spain",
    "PRT": "Portugal",
    "IRL": "Ireland",
    "NLD": "Netherlands",
}

INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "GDP growth (%)",
    "FP.CPI.TOTL.ZG": "Inflation rate (annual %)",
    "GC.DOD.TOTL.GD.ZS": "Government debt (% of GDP)",
    "SL.EMP.TOTL.ZS": "Employment rate (% of total labor force)",
    "SL.UEM.TOTL.ZS": "Unemployment rate (% of total labor force)"
}

START_YEAR = 2000
END_YEAR = 2025

# Fetch data
wb_dataset = []

for indicator_code, indicator_name in INDICATORS.items():
    df = fetch_indicator_data(
        indicator=indicator_code,
        countries=list(COUNTRIES.keys()),
        start_year=START_YEAR,
        end_year=END_YEAR
        # per_page = 2000,          # Parameters already set to default values in function
        # rate_limit_delay = 0.2    # Parameters already set to default values in function
    )

    df["indicator_name"] = indicator_name
    wb_dataset.append(df)

wb_dataset = pd.concat(wb_dataset, ignore_index=True)
wb_dataset["source"] = "wb_open_data"

# Reorder columns for better readability
wb_dataset = wb_dataset[["source", "country", "year", "indicator_name", "value"]]

print("World Bank Economic Indicators Data Sample:")
display(wb_dataset.head(10))

World Bank Economic Indicators Data Sample:


,source,country,year,indicator_name,value
0,wb_open_data,Malta,2024,GDP growth (%),6.799903
1,wb_open_data,Malta,2023,GDP growth (%),10.621519
2,wb_open_data,Malta,2022,GDP growth (%),2.483100
3,wb_open_data,Malta,2021,GDP growth (%),13.411704
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283
5,wb_open_data,Malta,2019,GDP growth (%),4.085056
6,wb_open_data,Malta,2018,GDP growth (%),7.189215
7,wb_open_data,Malta,2017,GDP growth (%),12.971342
8,wb_open_data,Malta,2016,GDP growth (%),4.078004
9,wb_open_data,Malta,2015,GDP growth (%),9.620703


### 2.3 Formatting World Bank API Output

Since retrieval models operate on text rather than raw numerical tables, each World Bank data row is converted into a natural-language sentence.

For example:
> **“In 2024, Malta’s GDP growth (%) was 6.79.”**

This transformation allows for structured indicators to be indexed and retrieved in the same way as unstructured text documents.

In [9]:
# Create natural language representation for each World Bank data row
def wb_row_to_text(row):
    return f"In {row['year']}, {row['country']}'s {row['indicator_name']} was {row['value']:.6f}."

wb_dataset["text"] = wb_dataset.apply(wb_row_to_text, axis=1)
display(wb_dataset.head(10))

,source,country,year,indicator_name,value,text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903."
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519."
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100."
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704."
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283."
5,wb_open_data,Malta,2019,GDP growth (%),4.085056,"In 2019, Malta's GDP growth (%) was 4.085056."
6,wb_open_data,Malta,2018,GDP growth (%),7.189215,"In 2018, Malta's GDP growth (%) was 7.189215."
7,wb_open_data,Malta,2017,GDP growth (%),12.971342,"In 2017, Malta's GDP growth (%) was 12.971342."
8,wb_open_data,Malta,2016,GDP growth (%),4.078004,"In 2016, Malta's GDP growth (%) was 4.078004."
9,wb_open_data,Malta,2015,GDP growth (%),9.620703,"In 2015, Malta's GDP growth (%) was 9.620703."


The generated natural language sentences are further processed using the same preprocessing function used for the Kaggle dataset.

In this case, year tokens are retained to support time-specific queries (e.g., “GDP growth in 2020”). This produces a sparse representation suitable for keyword-based retrieval while preventing the loss of temporal information.

In [10]:
wb_dataset["processed_text"] = wb_dataset["text"].apply(tokenize_and_filter, keep_years=True)
display(wb_dataset.head(10))

# Save fetched data to CSV
WBF_OUTPUT_PATH = DATA_TEMP_DIR / "wb_economic_indicators_dataset.csv"
wb_dataset.to_csv(WBF_OUTPUT_PATH, index=False)

,source,country,year,indicator_name,value,text,processed_text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903.",2024 malta gdp growth
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519.",2023 malta gdp growth
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100.",2022 malta gdp growth
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704.",2021 malta gdp growth
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283.",2020 malta gdp growth
5,wb_open_data,Malta,2019,GDP growth (%),4.085056,"In 2019, Malta's GDP growth (%) was 4.085056.",2019 malta gdp growth
6,wb_open_data,Malta,2018,GDP growth (%),7.189215,"In 2018, Malta's GDP growth (%) was 7.189215.",2018 malta gdp growth
7,wb_open_data,Malta,2017,GDP growth (%),12.971342,"In 2017, Malta's GDP growth (%) was 12.971342.",2017 malta gdp growth
8,wb_open_data,Malta,2016,GDP growth (%),4.078004,"In 2016, Malta's GDP growth (%) was 4.078004.",2016 malta gdp growth
9,wb_open_data,Malta,2015,GDP growth (%),9.620703,"In 2015, Malta's GDP growth (%) was 9.620703.",2015 malta gdp growth


### 2.4 Combining Datasets

In [11]:
print("Kaggle FSA Processed Data Sample:")
display(kaggle_dataset.head())
print("World Bank Economic Indicators Data Sample:")
display(wb_dataset.head())

Kaggle FSA Processed Data Sample:


,source,text,processed_text,year,sentiment
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,NaN,positive
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,NaN,negative
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,2010.0,positive
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,NaN,neutral
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,NaN,neutral


World Bank Economic Indicators Data Sample:


,source,country,year,indicator_name,value,text,processed_text
0,wb_open_data,Malta,2024,GDP growth (%),6.799903,"In 2024, Malta's GDP growth (%) was 6.799903.",2024 malta gdp growth
1,wb_open_data,Malta,2023,GDP growth (%),10.621519,"In 2023, Malta's GDP growth (%) was 10.621519.",2023 malta gdp growth
2,wb_open_data,Malta,2022,GDP growth (%),2.483100,"In 2022, Malta's GDP growth (%) was 2.483100.",2022 malta gdp growth
3,wb_open_data,Malta,2021,GDP growth (%),13.411704,"In 2021, Malta's GDP growth (%) was 13.411704.",2021 malta gdp growth
4,wb_open_data,Malta,2020,GDP growth (%),-3.457283,"In 2020, Malta's GDP growth (%) was -3.457283.",2020 malta gdp growth


Before merging the datasets, empty metadata fields are added to the Kaggle dataset to ensure schema consistency across both data sources.

Both datasets are then reordered to follow a shared column structure:

- `source`
- `text`
- `processed_text`
- `country`
- `year`
- `indicator_name`
- `value`

The aligned datasets are concatenated into a single unified table, forming the final dataset used throughout the RAG pipeline. A sample of the merged data is displayed to verify correct integration.

The resulting dataset is saved to disk as a CSV file and represents the complete RAG corpus for this project, combining unstructured textual data with structured economic metadata from multiple sources.


In [12]:
# In order to keep metadata consistent across both datasets, we add empty columns to the Kaggle dataset
kaggle_dataset["country"] = None
kaggle_dataset["indicator_name"] = None
kaggle_dataset["value"] = None

# Final reordering of columns
kaggle_dataset = kaggle_dataset[
    ["source", "text", "processed_text", "country", "year", "indicator_name", "value"]
]

wb_dataset = wb_dataset[
    ["source", "text", "processed_text", "country", "year", "indicator_name", "value"]
]

# Combine both datasets into final RAG dataset
final_rag_dataset = pd.concat([kaggle_dataset, wb_dataset], ignore_index=True)

print("Final RAG Dataset Sample:")
display(final_rag_dataset.head(20))

# Save final RAG dataset to CSV
RAG_OUTPUT_PATH = DATA_OUTPUT_DIR / "final_rag_dataset.csv"
final_rag_dataset.to_csv(RAG_OUTPUT_PATH, index=False)

Final RAG Dataset Sample:


C:\Users\gianl\AppData\Local\Temp\ipykernel_2088\3269066736.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_rag_dataset = pd.concat([kaggle_dataset, wb_dataset], ignore_index=True)


,source,text,processed_text,country,year,indicator_name,value
0,kaggle,The GeoSolutions technology will leverage Bene...,geosolutions technology leverage benefon gps s...,None,NaN,None,NaN
1,kaggle,"$ESI on lows, down $1.50 to $2.50 BK a real po...",esi lows bk real possibility,None,NaN,None,NaN
2,kaggle,"For the last quarter of 2010 , Componenta 's n...",last quarter 2010 componenta net sales doubled...,None,2010.0,None,NaN
3,kaggle,According to the Finnish-Russian Chamber of Co...,according chamber commerce major construction ...,None,NaN,None,NaN
4,kaggle,The Swedish buyout firm has sold its remaining...,swedish buyout firm sold remaining percent sta...,None,NaN,None,NaN
5,kaggle,$SPY wouldn't be surprised to see a green close,spy would surprised see green close,None,NaN,None,NaN
6,kaggle,Shell's $70 Billion BG Deal Meets Shareholder ...,shell billion bg deal meets shareholder skepti...,None,NaN,None,NaN
7,kaggle,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,ssh communications security corp stock exchang...,None,2008.0,None,NaN
8,kaggle,Kone 's net sales rose by some 14 % year-on-ye...,kone net sales rose first nine months 2008,None,2008.0,None,NaN
9,kaggle,The Stockmann department store will have a tot...,stockmann department store total floor space s...,None,NaN,None,NaN


## 3. Getting to know the data

This section provides an exploratory analysis of the final RAG dataset, with the main aim being to understand the structure and contents of the dataset before applying retrieval and generation methods.

### 3.1 Distribution of Data Sources

This subsection looks at the distribution of records across the different data sources included in the final dataset. A histogram is used to visualise the relative contribution of each source, helping to identify any potential imbalance between sources.

In [13]:
fig = px.histogram(
    final_rag_dataset,
    x="source",
    color="source",
    text_auto=True,
    title="Distribution of Data Sources",
    labels={"source": "Data Source"},
    color_discrete_sequence=["#AEC6CF", "#FF6961"]  # pastel blue & pastel pink
)

fig.show()

### 3.2 Missing Values per Column

This subsection analyses missing values across all columns in the dataset. The total number of missing entries per column is calculated and visualised to highlight which fields are sparsely populated.

This analysis helps distinguish between expected missing values (e.g. structured metadata not applicable to all records) and potential data quality issues that may require attention in later stages.

In [14]:
missing_df = final_rag_dataset.isna().sum().reset_index()
missing_df = missing_df[missing_df[0] > 0].sort_values(by=0, ascending=True)

if not missing_df.empty:
    missing_df.columns = ["column", "missing_count"]

    fig = px.bar(
        missing_df,
        x="column",
        y="missing_count",
        text_auto=True,
        title="Missing Values per Column",
        labels={"missing_count": "Number of Missing Values"},
        color="column",
        color_discrete_sequence=[
            "#AEC6CF",  # pastel blue
            "#FF6961",  # pastel pink
            "#C1E1C1",  # pastel green
            "#D7BDE2",  # pastel purple
            "#FFF2CC",  # pastel yellow
            "#FADADD",  # pastel peach
            "#D7B4F3"   # pastel lavender
        ]
    )

    fig.show()

### 3.3 Temporal Coverage (Year Distribution)

This subsection explores the temporal coverage of the dataset by analysing the distribution of available year metadata. Records without a year value are excluded from this analysis.

A histogram with an overlaid density curve is used to illustrate how data points are distributed over time. Summary statistics, including the mean, median, and mode, are visualised to provide additional insight into the central tendency and temporal concentration of the dataset.

As show by the plot, the data is mostly from 2005-2010, peaking around 2008–2009. The mean, median, and mode are close, showing that the data is well-centred in that period, with fewer records in earlier and later years. This means the dataset is temporally imbalanced and mainly reflects the late 2000s.

In [15]:
data = final_rag_dataset.dropna(subset=["year"])
years = data["year"].astype(int)

# Histogram
fig = px.histogram(
    data,
    x="year",
    nbins=30,
    histnorm="probability density",
    title="Distribution of Years with Density Curve",
    labels={"year": "Year"}
)


fig.update_traces(marker_color="#AEC6CF")

# KDE line
kde = gaussian_kde(years)
x = np.linspace(years.min(), years.max(), 1000)

fig.add_trace(
    go.Scatter(
        x=x,
        y=kde(x),
        mode="lines",
        name="Density Curve",
        line=dict(color="#FF6961", width=2)  # pastel pink
    )
)

# Adding mean, median, mode lines
stats = {
    "Mean": (years.mean(), "green"),
    "Median": (years.median(), "blue"),
    "Mode": (years.mode().iloc[0], "orange"),
}

for label, (value, color) in stats.items():
    fig.add_vline(
        x=value,
        line=dict(color=color, dash="dash"),
        name=f"{label} ({value:.1f})",
        showlegend=True
    )

fig.show()

### 3.4 Text Length Distribution

This subsection analyses the distribution of raw text lengths across the dataset. Text length is measured as the number of words per document.

A probability density histogram and kernel density estimate are used to characterise the overall shape of the distribution. Mean, median, and mode values are included to highlight typical document lengths and identify potential outliers or extreme cases.

The majority of texts are short, with the highest frequency around 10 words. The mean (~19.6) is higher than the median (18), indicating a right-skewed distribution caused by a smaller number of longer texts

In [16]:
temp = final_rag_dataset.copy()
temp["text_length"] = final_rag_dataset["text"].str.split().str.len()

fig = px.histogram(
    temp,
    x="text_length",
    nbins=50,
    histnorm="probability density",
    title="Distribution of Text Lengths",
    labels={"text_length": "Number of Words"}
)

fig.update_traces(marker_color="#AEC6CF")

# KDE line
kde = gaussian_kde(temp["text_length"])
x = np.linspace(temp["text_length"].min(), temp["text_length"].max(), 1000)

fig.add_trace(
    go.Scatter(
        x=x,
        y=kde(x),
        mode="lines",
        name="Density Curve",
        line=dict(color="#FF6961", width=2)  # pastel pink
    )
)

# Adding mean, median, mode lines
stats = {
    "Mean": (temp["text_length"].mean(), "green"),
    "Median": (temp["text_length"].median(), "blue"),
    "Mode": (temp["text_length"].mode().iloc[0], "orange"),
}

for label, (value, color) in stats.items():
    fig.add_vline(
        x=value,
        line=dict(color=color, dash="dash"),
        name=f"{label} ({value:.1f})",
        showlegend=True
    )

fig.show()

### 3.5 Processed Text Length Distribuution

This subsection examines the length distribution of the processed text representation. As with the raw text analysis, text length is measured in terms of word count.

Comparing this distribution with the raw text length provides insight into the impact of preprocessing steps such as stopword removal and token filtering, and helps assess how much textual compression is introduced by the sparse preprocessing pipeline.

Processed texts are significantly shorter than the raw texts, with most entries containing 7–10 words. The mean (~10.4) is slightly higher than the median (9), showing a right-skewed distribution caused by a small number of longer processed texts. This is primarily due to the effect of stopword removal and token filtering, which substantially reduces text length.

In [17]:
temp = final_rag_dataset.copy()
temp["pt_length"] = final_rag_dataset["processed_text"].str.split().str.len()

fig = px.histogram(
    temp,
    x="pt_length",
    nbins=50,
    histnorm="probability density",
    title="Distribution of Processed Text Lengths",
    labels={"pt_length": "Number of Words"}
)

fig.update_traces(marker_color="#AEC6CF")

# KDE line
kde = gaussian_kde(temp["pt_length"])
x = np.linspace(temp["pt_length"].min(), temp["pt_length"].max(), 1000)

fig.add_trace(
    go.Scatter(
        x=x,
        y=kde(x),
        mode="lines",
        name="Density Curve",
        line=dict(color="#FF6961", width=2)  # pastel pink
    )
)

# Adding mean, median, mode lines
stats = {
    "Mean": (temp["pt_length"].mean(), "green"),
    "Median": (temp["pt_length"].median(), "blue"),
    "Mode": (temp["pt_length"].mode().iloc[0], "orange"),
}

for label, (value, color) in stats.items():
    fig.add_vline(
        x=value,
        line=dict(color=color, dash="dash"),
        name=f"{label} ({value:.1f})",
        showlegend=True
    )

fig.show()

### 3.6 Processed vs Raw Text Length

This plot compares the length of the raw text with its corresponding processed representation. Each point represents a single record in the dataset. The scatter plot highlights the relationship between raw and processed text lengths and illustrates the reduction in text size introduced by stopword removal and token filtering. This comparison helps quantify the impact of sparse preprocessing on the original textual content.

The scatter plot shows a strong positive relationship between raw and processed text lengths, which is further highlighted by the fitted linear trend line. Processed texts are consistently shorter due to preprocessing, while the relative ordering of document lengths is largely preserved.

In [18]:
temp = final_rag_dataset.copy()
temp["text_length"] = temp["text"].str.split().str.len()
temp["processed_length"] = temp["processed_text"].str.split().str.len()

fig = px.scatter(
    temp,
    x="text_length",
    y="processed_length",
    title="Processed vs Raw Text Length",
    labels={
        "text_length": "Raw Text Length (words)",
        "processed_length": "Processed Text Length (words)"
    },
    opacity=0.6,
    trendline="ols"
)

fig.show()

### 3.7 Text Length by Data Source (Raw vs Processed)

This subsection compares text length distributions across data sources for both raw and processed text representations. For each representation, boxplots and violin plots are used to visualise differences in central tendency, spread, and distribution shape between sources. The boxplots provide a summary of typical text lengths and highlight the presence of outliers, while the violin plots reveal the distribution density and variability.

For raw text, the Kaggle dataset shows more variability and a wider range of text lengths compared to the World Bank data, which is more tightly clustered around shorter values. This reflects the unstructured nature of the Kaggle data versus the more uniform structure of the World Bank records. After preprocessing, text lengths are reduced across both sources, with a noticable reduction for the Kaggle dataset. These results highlight how sparse preprocessing affects datasets differently and underline the importance of source-aware retrieval and chunking strategies.

In [19]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Text Length Distribution by Source (Boxplot)",
        "Text Length Distribution by Source (Violin Plot)"
    ]
)

# Boxplot
for src in temp["source"].unique():
    fig.add_trace(
        go.Box(
            x=temp[temp["source"] == src]["source"],
            y=temp[temp["source"] == src]["text_length"],
            name=src,
            showlegend=False
        ),
        row=1,
        col=1
    )

# Violin plot
for src in temp["source"].unique():
    fig.add_trace(
        go.Violin(
            x=temp[temp["source"] == src]["source"],
            y=temp[temp["source"] == src]["text_length"],
            name=src,
            box_visible=True,
            meanline_visible=True,
            showlegend=False
        ),
        row=1,
        col=2
    )

fig.update_layout(
    title="Processed Text Length Distribution by Data Source",
    yaxis_title="Number of Words",
)

fig.show() 

In [20]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Processed Text Length by Source (Boxplot)",
        "Processed Text Length by Source (Violin Plot)"
    ],
    shared_yaxes=True
)

# Boxplot
for src in temp["source"].unique():
    fig.add_trace(
        go.Box(
            x=temp[temp["source"] == src]["source"],
            y=temp[temp["source"] == src]["processed_length"],
            name=src,
            showlegend=True
        ),
        row=1,
        col=1
    )

# Violin plot
for src in temp["source"].unique():
    fig.add_trace(
        go.Violin(
            x=temp[temp["source"] == src]["source"],
            y=temp[temp["source"] == src]["processed_length"],
            name=src,
            box_visible=True,
            meanline_visible=True,
            showlegend=False
        ),
        row=1,
        col=2
    )

fig.update_layout(
    title="Processed Text Length Distribution by Data Source",
    yaxis_title="Number of Words"
)

fig.show()

## 4. Sparse Retrieval Methods

In [ ]:
final_rag_dataset = pd.read_csv('Datasets/Outputs/final_rag_dataset.csv')

#Calculating BM25 Score

def termFrequency(term, document):
    termCount = document.count(term)
    docLength = len(document)
    
    return (termCount / (termCount + k*(1 - b + b * (docLength / avgDocLength))))

def inverseDocFrequency(term):
    docsContainingTerm = final_rag_dataset['processed_text'].str.contains(term).sum()

    return math.log((totalCorpLength - docsContainingTerm + 0.5) / (docsContainingTerm + 0.5))

def bm25Score(query, document):
    bmScore = 0
    for term in query:
        tf = termFrequency(term, document)
        idf = inverseDocFrequency(term)
        bmScore += tf*idf
    return bmScore

k = 1.2
b = 0.75
avgDocLength = final_rag_dataset['processed_text'].str.split().str.len().mean()
totalCorpLength = len(final_rag_dataset)

query = "euro venezuela"

cleanedQuery = ''.join(char for char in query if char.isalnum() or char.isspace())
processedQuery = word_tokenize(cleanedQuery.lower())
for doc in final_rag_dataset['processed_text']:
    processedDoc = doc.split()
    score = bm25Score(processedQuery, processedDoc)
    if score != 0:
        print(score)


['new', 'agreement', 'expands', 'cooperation', 'companies', 'involves', 'transfer', 'certain', 'engineering', 'documentation', 'functions', 'larox', 'etteplan']
['geosolutions', 'technology', 'leverage', 'benefon', 'gps', 'solutions', 'providing', 'location', 'based', 'search', 'technology', 'communities', 'platform', 'location', 'relevant', 'multimedia', 'content', 'new', 'powerful', 'commercial', 'model']
['esi', 'lows', 'bk', 'real', 'possibility']
['last', 'quarter', '2010', 'componenta', 'net', 'sales', 'doubled', 'period', 'year', 'earlier', 'moved', 'zero', 'profit', 'loss']
['according', 'chamber', 'commerce', 'major', 'construction', 'companies', 'finland', 'operating', 'russia']
['swedish', 'buyout', 'firm', 'sold', 'remaining', 'percent', 'stake', 'almost', 'eighteen', 'months', 'taking', 'company', 'public', 'finland']
['spy', 'would', 'surprised', 'see', 'green', 'close']
['shell', 'billion', 'bg', 'deal', 'meets', 'shareholder', 'skepticism']
['ssh', 'communications', 'se

## 5. Dense Retrieval Methods